# 03 - Huấn luyện M3 trên Colab (Phase 3C)
Chạy tuần tự từ trên xuống. Sửa các đường dẫn ở ô 1 cho đúng với Drive của bạn (đây là GIẢ ĐỊNH, mình chưa thấy tên file thật).

In [ ]:
# Ô 1 - cấu hình (SỬA CHO ĐÚNG)
DATA_TAR = "/content/drive/MyDrive/music-copyright-ai/data.tar"   # chứa data/interim/logmel, data/processed/tracks.parquet, ...
CODE_TAR = "/content/drive/MyDrive/music-copyright-ai/code.tar"   # chứa configs/ và src/ BẢN MỚI NHẤT (có 3C), tar lại mỗi khi sửa code
OUT_DIR  = "/content/drive/MyDrive/music-copyright-ai/checkpoints" # checkpoint ghi thẳng lên Drive để không mất khi Colab ngắt
SEED = 42

In [ ]:
# Ô 2 - Drive + GPU
from google.colab import drive
drive.mount("/content/drive")
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime > Change runtime type > T4 GPU"

In [ ]:
# Ô 3 - giải nén ra ổ cục bộ (tránh I/O chậm qua Drive FUSE); code.tar ghi đè sau cùng
!rm -rf /content/repo && mkdir -p /content/repo
!tar -xf "$DATA_TAR" -C /content/repo
!tar -xf "$CODE_TAR" -C /content/repo
%cd /content/repo
!ls

In [ ]:
# Ô 4 - phụ thuộc còn thiếu (torch đã có sẵn trên Colab)
!pip install -q pytest pyarrow pyyaml matplotlib pandas

In [ ]:
# Ô 5 - test 3A+3B+3C (mong đợi 36 passed)
!python -m pytest tests/test_m3_data_3a.py tests/test_m3_model_3b.py tests/test_train_3c.py -q

In [ ]:
# Ô 6 - chạy thử 5 step để kiểm tra chạy được và đo tốc độ (ghi cột "seconds" lại cho mình)
!python -m src.train_m3 --seed $SEED --epochs 1 --max-steps 5 --preload --out-dir /content/smoke_run

In [ ]:
# Ô 7 - HUẤN LUYỆN ĐẦY ĐỦ. An toàn khi Colab ngắt: chạy lại đúng ô này, --resume sẽ tiếp tục từ last.pt
!python -m src.train_m3 --seed $SEED --preload --resume --out-dir "$OUT_DIR/m3_seed$SEED"

In [ ]:
# Ô 8 - learning curve và tóm tắt
from IPython.display import Image, display
import json
display(Image(f"{OUT_DIR}/m3_seed{SEED}/curve.png"))
print(json.dumps(json.load(open(f"{OUT_DIR}/m3_seed{SEED}/summary.json")), indent=2, ensure_ascii=False))

In [ ]:
# Ô 9 - chép kết quả về Drive (best.pt, last.pt đã ở sẵn trên Drive); JSON kết quả để chép về repo local
!mkdir -p "$OUT_DIR/results" && cp experiments/results/e3_train_seed$SEED.json "$OUT_DIR/results/" && cp data/interim/m3_norm.json "$OUT_DIR/results/"
!ls -la "$OUT_DIR/m3_seed$SEED" "$OUT_DIR/results"

## Đánh giá trên Val (Phase 3D-2)
Cần `code.tar` bản mới (có `src/models/m3_retrieval.py`, `run_baselines.py`, `run_robustness.py` đã sửa). Không đọc Test.

In [ ]:
# Ô 10 - Recall/ROC trên truy vấn sạch Val, rồi bảng robustness 17 cấu hình x 7 nhóm
CKPT = f"{OUT_DIR}/m3_seed{SEED}/best.pt"
!python -m src.run_baselines --split val --models m3 --m3-ckpt "$CKPT" --tag seed$SEED
!python -m src.run_robustness eval --split val --models m3 --m3-ckpt "$CKPT" --tag seed$SEED

In [ ]:
# Ô 11 - chép kết quả về Drive để tải về repo local (experiments/results/)
!cp experiments/results/e1_m3_seed${SEED}_val.json experiments/results/e5_robust_val_m3_seed${SEED}*.json experiments/results/e5_robust_val_m3_seed${SEED}_perquery.npz "$OUT_DIR/results/"
!ls -la "$OUT_DIR/results"

## 3E - nhiều seed, ablation, chẩn đoán nhiễu
Mỗi lần huấn luyện khoảng 19 phút trên T4; mọi lệnh đều `--resume` được, chạy lại ô nếu Colab ngắt. Chỉ dùng Val.

In [ ]:
# Ô 12 - thêm 2 seed (khoảng 40 phút) + đánh giá Val từng seed
for s in (43, 44):
    !python -m src.train_m3 --seed {s} --preload --resume --out-dir "{OUT_DIR}/m3_seed{s}"
    !python -m src.run_robustness eval --split val --models m3 --m3-ckpt "{OUT_DIR}/m3_seed{s}/best.pt" --tag seed{s}

In [ ]:
# Ô 13 - chẩn đoán nhiễu (vài phút): nhiễu thật so với nhiễu tổng hợp ở miền log-mel
!python -m src.analysis.m3_noise_check --ckpt "{OUT_DIR}/m3_seed42/best.pt" --tag seed42

In [ ]:
# Ô 14 - ablation ưu tiên (4 biến thể, khoảng 80 phút). Đổi danh sách nếu muốn; tên có trong configs/m3_ablations.yaml
ABL = ["no_noise", "no_pitch_tempo", "gap", "no_overlap"]
for a in ABL:
    !python -m src.train_m3 --seed 42 --ablation {a} --preload --resume --out-dir "{OUT_DIR}/m3_abl_{a}_seed42"
    !python -m src.run_robustness eval --split val --models m3 --m3-ckpt "{OUT_DIR}/m3_abl_{a}_seed42/best.pt" --tag abl_{a}_seed42

In [ ]:
# Ô 15 - tổng hợp và chép kết quả về Drive
!python -m src.evaluation.summarize_m3
!cp experiments/results/e3_*.json experiments/results/e5_robust_val_m3_*.json "$OUT_DIR/results/"
!cp reports/tables/m3_val_summary.md "$OUT_DIR/results/"
!ls "$OUT_DIR/results"

## 3E-2 - kiểm chứng mô phỏng nhiễu kết hợp (chỉ chạy sau khi các ô 12-15 đang chạy xong, và sau khi đã tar lại code mới)

In [ ]:
# Ô 16 - chạy lại chẩn đoán nhiễu với hai mô phỏng (cộng công suất và coherent), thêm lưới SNR (khoảng 4 phút)
!python -m src.analysis.m3_noise_check --ckpt "{OUT_DIR}/m3_seed42/best.pt" --tag seed42_v2

In [ ]:
# Ô 17 - CHỈ chạy nếu Ô 16 cho thấy R@1 nhiễu coherent gần R@1 nhiễu thật (xem guide mục 17). ~19 phút
!python -m src.train_m3 --seed 42 --ablation noise_coherent --preload --resume --out-dir "{OUT_DIR}/m3_abl_noise_coherent_seed42"
!python -m src.run_robustness eval --split val --models m3 --m3-ckpt "{OUT_DIR}/m3_abl_noise_coherent_seed42/best.pt" --tag abl_noise_coherent_seed42
!python -m src.evaluation.summarize_m3

## 3E-3 - hiệu chuẩn k của nhiễu coherent (sau khi tar lại code mới)

In [ ]:
# Ô 18 - đo R@1 của nhiễu coherent với k = 2,4,8,16 so với nhiễu thật, trên checkpoint seed 42 có sẵn (khoảng 6-8 phút)
!python -m src.analysis.m3_noise_check --ckpt "{OUT_DIR}/m3_seed42/best.pt" --tag seed42_v3

In [ ]:
# Ô 19 - huấn luyện với k tốt nhất (đặt K theo dòng "k tốt nhất" của Ô 18; k=2,4,8 đã có sẵn tên ablation noise_coherent_k{K})
K = 4
!python -m src.train_m3 --seed 42 --ablation noise_coherent_k{K} --preload --resume --out-dir "{OUT_DIR}/m3_abl_noise_coherent_k{K}_seed42"
!python -m src.run_robustness eval --split val --models m3 --m3-ckpt "{OUT_DIR}/m3_abl_noise_coherent_k{K}_seed42/best.pt" --tag abl_noise_coherent_k{K}_seed42
!python -m src.evaluation.summarize_m3

## 3F - chốt cấu hình cuối: thêm seed 43, 44 cho `noise_coherent_k2` (khoảng 60 phút, mỗi seed ~30 phút vì sinh nhiễu coherent tốn CPU)

In [ ]:
# Ô 20 - huấn luyện + đánh giá Val seed 43, 44 của cấu hình cuối; chạy lại ô nếu Colab ngắt (--resume)
for s in (43, 44):
    !python -m src.train_m3 --seed {s} --ablation noise_coherent_k2 --preload --resume --out-dir "{OUT_DIR}/m3_abl_noise_coherent_k2_seed{s}"
    !python -m src.run_robustness eval --split val --models m3 --m3-ckpt "{OUT_DIR}/m3_abl_noise_coherent_k2_seed{s}/best.pt" --tag abl_noise_coherent_k2_seed{s}
!cp experiments/results/e3_*.json experiments/results/e5_robust_val_m3_*.json "$OUT_DIR/results/"
!ls "$OUT_DIR/results" | grep noise_coherent_k2

## 3F - khôi phục kết quả seed 42 của cấu hình cuối (Ô 19 trước đây không chép kết quả về Drive)

In [ ]:
# Ô 21 - đánh giá lại seed 42 từ best.pt có sẵn trên Drive (vài phút) và chép đủ file kết quả về Drive
CK = f"{OUT_DIR}/m3_abl_noise_coherent_k2_seed42"
!python -m src.run_robustness eval --split val --models m3 --m3-ckpt "{CK}/best.pt" --tag abl_noise_coherent_k2_seed42
!cp "{CK}/summary.json" experiments/results/e3_train_abl_noise_coherent_k2_seed42.json
!cp experiments/results/e3_*.json experiments/results/e5_robust_val_m3_*.json "$OUT_DIR/results/"
!ls "$OUT_DIR/results" | grep noise_coherent_k2_seed42